In [8]:
# import packages
import pandas as pd
import numpy as np

## Collaborative filtering functions from Homework 3

In [4]:
def user_collab_filter(df, target_user, similarity='Cosine', k=5):
    # check if the user is in the dataframe
    if target_user not in df.index:
        print(f"Error: {target_user} not found in the dataset.")
        return None

    # check if the user has missing ratings
    if not df.loc[target_user].isna().any():
        print(f"Error: {target_user} has no missing ratings.")
        return None

    # compute each user's average rating in the original dataframe
    user_means = df.mean(axis=1, skipna=True)

    df_avg = df.copy()
    # compute the mean of each row and fill NaN values with the mean
    for idx, row in df_avg.iterrows():
        row_mean = row.mean()
        if pd.isna(row_mean):
            df_avg.loc[idx] = row.fillna(0)
        else:
            df_avg.loc[idx] = row.fillna(row_mean)

    # center each row
    df_centered = df_avg.copy()
    for idx, row in df_centered.iterrows():
        row_mean = row.mean()
        df_centered.loc[idx] -= row_mean

    # find similarity between each user
    similarity_series = pd.Series(index=df.index, dtype=float)
    target_user_row = df_centered.loc[target_user]
    x = target_user_row.to_numpy()
    for idx, row in df_centered.iterrows():
        y = row.to_numpy()
        if similarity == 'Cosine':
            denom = np.linalg.norm(x) * np.linalg.norm(y)
            similarity_score = 0.0 if denom == 0 else np.dot(x, y) / denom
        elif similarity == 'L2':
            similarity_score = -np.linalg.norm(x - y)
        else:
            print(f"Error: {similarity} is not a valid similarity. Please use Cosine or L2.")
            return None
        similarity_series.loc[idx] = similarity_score

    # drop the user's similarity score to themselves
    similarity_series = similarity_series.drop(target_user)

    # min-max scale the similarity series
    similarity_min_max = (similarity_series - similarity_series.min()) / (similarity_series.max() - similarity_series.min())

    # find the k most similar users
    similarity_kmost = similarity_min_max.nlargest(k)
    users_kmost = similarity_kmost.index

    # find empty ratings for the target user
    empty_item_list = df.columns[df.loc[target_user].isna()].tolist()

    # predict ratings using k most similar users
    predicted_rating = {}
    for item in empty_item_list:
        similar_ratings = df.loc[users_kmost, item]
        similar_ratings = similar_ratings.fillna(user_means.loc[users_kmost])

        num = (similarity_kmost * similar_ratings).sum()
        denom = similarity_kmost.sum()
        
        if denom == 0:
            predicted_rating[item] = float(np.nan)
        else:
            predicted_rating[item] = float(num / denom)

    return predicted_rating

def item_collab_filter(df, target_user, similarity='Cosine', k=5):
    # check if the user is in the dataframe
    if target_user not in df.index:
        print(f"Error: {target_user} not found in the dataset.")
        return None

    # check if the user has missing ratings
    if not df.loc[target_user].isna().any():
        print(f"Error: {target_user} has no missing ratings.")
        return None

    # compute each item's average rating in the original dataframe
    item_means = df.mean(axis=0, skipna=True)

    df_centered = df.copy()
    # center each column
    for col in df_centered.columns:
        col_mean = df_centered[col].mean()
        df_centered[col] = df_centered[col] - col_mean

    # find empty ratings for the target user
    empty_item_list = df.columns[df.loc[target_user].isna()].tolist()

    # predict ratings using k most similar items
    predicted_rating = {}
    for item in empty_item_list:
        # find similarity between each item
        similarity_series = pd.Series(index=df.columns, dtype=float)
        x_full = df_centered[item]

        for col in df_centered.columns:
            y_full = df_centered[col]

            corated_mask = x_full.notna() & y_full.notna()
            x = x_full[corated_mask].to_numpy()
            y = y_full[corated_mask].to_numpy()

            if len(x) == 0:
                similarity_score = 0.0
            elif similarity == 'Cosine':
                denom = np.linalg.norm(x) * np.linalg.norm(y)
                similarity_score = 0.0 if denom == 0 else np.dot(x, y) / denom
            elif similarity == 'L2':
                similarity_score = -np.linalg.norm(x - y)
            else:
                print(f"Error: {similarity} is not a valid similarity. Please use Cosine or L2.")
                return None

            similarity_series.loc[col] = similarity_score

        # drop the item's similarity score to itself
        similarity_series = similarity_series.drop(item)

        # min-max scale the similarity series
        similarity_min_max = (similarity_series - similarity_series.min()) / (similarity_series.max() - similarity_series.min())

        # find the k most similar items
        similarity_kmost = similarity_min_max.nlargest(k)
        users_kmost = similarity_kmost.index

        similar_ratings = df.loc[target_user, users_kmost]
        similar_ratings = similar_ratings.fillna(item_means.loc[users_kmost])

        num = (similarity_kmost * similar_ratings).sum()
        denom = similarity_kmost.sum()

        if denom == 0:
            predicted_rating[item] = float(np.nan)
        else:
            predicted_rating[item] = float(num / denom)

    return predicted_rating

def collab_filter(filter_type, df, target_user, similarity='Cosine', k=5):
    if filter_type == 'User':
        return user_collab_filter(df, target_user, similarity, k)
    elif filter_type == 'Item':
        return item_collab_filter(df, target_user, similarity, k)
    else:
        print(f"Error: {filter_type} not found a valid filter. Please use User or Item.")
        return None

In [5]:
sandwiches = {
    'bs_0': 'Ham, Egg, and Cheese',
    'bs_1': 'Sausage, Egg, and Cheese',
    'bs_2': 'Double Egg and Cheese',
    'bs_3': 'Eggs on the Beach',
    'bs_4': 'Chicken, sausage, egg and cheese',
    'bs_5': 'Bacon, egg and cheese',
    'bs_6': 'Vegan Breakfast Sandwich',
    's_0': 'The Hemenway',
    's_1': 'The Fenway',
    's_2': 'The Northeastern',
    's_3': 'The Huntington',
    's_4': 'The Huskie',
    's_5': 'Marino Fitness',
    's_6': 'The Boston Connection',
    's_7': 'Chicken Salad',
    's_8': 'Game of Thrones',
    's_9': 'George Likes his Chicken Spicy',
    's_10': 'Phi Gamma Melta',
    's_11': 'Steak and Cheese',
    's_12': 'The Symphony',
    's_13': 'Tuna Salad',
    's_14': 'Buffalo Chicken Tender Sub',
    's_15': 'Gobbler',
    's_16': 'Lighten Up Francis',
    's_17': 'Meatball Sub',
    's_18': 'Notorious DTD',
    's_19': 'The Pike',
    's_20': 'Healthy Choice'
}

## Creating Synthetic Data for Collaborative Filtering Model 1

In [10]:
n_students = 100
fill_fraction = 1/3
rng = np.random.default_rng(42)

cols = list(sandwiches.keys())
rows = [f'stu_{i}' for i in range(n_students)]

ratings = rng.integers(1, 6, size=(n_students, len(cols))).astype(float)

filled_mask = rng.random((n_students, len(cols))) < fill_fraction

data = np.where(filled_mask, ratings, np.nan)

df_ratings = pd.DataFrame(data, index=rows, columns=cols)

df_ratings.head()

,bs_0,bs_1,bs_2,bs_3,bs_4,bs_5,bs_6,s_0,s_1,s_2,...,s_11,s_12,s_13,s_14,s_15,s_16,s_17,s_18,s_19,s_20
stu_0,NaN,4.0,NaN,3.0,3.0,5.0,NaN,4.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,5.0
stu_1,NaN,NaN,NaN,NaN,1.0,3.0,5.0,NaN,5.0,NaN,...,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,1.0
stu_2,3.0,NaN,NaN,NaN,NaN,NaN,2.0,5.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN,5.0
stu_3,NaN,4.0,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN,2.0,4.0
stu_4,NaN,NaN,NaN,3.0,1.0,NaN,2.0,1.0,3.0,NaN,...,3.0,3.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN


In [11]:
collab_filter('Item', df_ratings, 'stu_50')

{'bs_0': 2.610093930337242,
 'bs_1': 3.0329774322058562,
 'bs_3': 3.203730070354907,
 's_0': 3.2055664780002813,
 's_4': 2.6684624803513484,
 's_5': 2.6288099785098322,
 's_8': 2.593974460978921,
 's_10': 2.974348846218163,
 's_11': 1.7685875031621336,
 's_12': 3.437631530343722,
 's_14': 3.0229573697966146,
 's_16': 2.1588290427005625,
 's_17': 1.9204288642820715,
 's_18': 2.91006622462991,
 's_19': 2.473353196633557}